# EMG-Based Classification of Standing Postures

This notebook implements a DeepConvNet-based model to classify standing postures using EMG signals recorded from different muscles (soleus (SOL) and flexor digitorum brevis (FDB)).

The primary objective is to compare muscle-specific contributions to posture classification (four postures) and determine whether one muscle provides more posture-discriminative information than the other.

## Notes before running

- The original human-subject EMG data are not included in this repository.
- Place the `.mat` files in the `data/` folder.
- The expected MATLAB variables are `datasets` for EMG windows and `labelss` for posture labels.
- Labels are assumed to be coded as 1–4 and are converted to 0–3 for TensorFlow.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix
from tensorflow.keras import Model
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.layers import (
    Activation, BatchNormalization, Conv2D, Dense, Dropout,
    Flatten, Input, MaxPooling2D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

POSTURE_NAMES = ["Bipedal", "One-leg", "Tandem", "Tiptoe"]

In [ ]:
MUSCLE = "SOL"  # use "SOL" or "FDB"
DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_CHANNELS = 64
N_SAMPLES = 512
N_CLASSES = 4
WINDOW_LABEL = "Time_W0.25O0.5"

EPOCHS = 100
BATCH_SIZE = 64
DROPOUT_RATE = 0.25
LEARNING_RATE = 1e-4
SEED = 42

In [ ]:
def set_random_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def load_mat_variable(file_path, variable_name):
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    mat = sio.loadmat(file_path)
    if variable_name not in mat:
        available = [key for key in mat.keys() if not key.startswith("__")]
        raise KeyError(
            f"Variable '{variable_name}' was not found in {file_path.name}. "
            f"Available variables: {available}"
        )
    return mat[variable_name]


def load_emg_dataset(data_dir, muscle, n_channels=64, n_samples=512, window_label="Time_W0.25O0.5"):
    muscle = muscle.upper()
    train_data_file = data_dir / f"dataset12_{muscle}_{window_label}train.mat"
    test_data_file = data_dir / f"dataset12_{muscle}_{window_label}test.mat"
    train_label_file = data_dir / f"labels12_{muscle}_{window_label}train.mat"
    test_label_file = data_dir / f"labels12_{muscle}_{window_label}test.mat"

    x_train = load_mat_variable(train_data_file, "datasets")
    x_test = load_mat_variable(test_data_file, "datasets")
    y_train = load_mat_variable(train_label_file, "labelss")
    y_test = load_mat_variable(test_label_file, "labelss")

    x_train = x_train.reshape((-1, n_channels, n_samples, 1)).astype("float32")
    x_test = x_test.reshape((-1, n_channels, n_samples, 1)).astype("float32")
    y_train = np.asarray(y_train).reshape(-1).astype(int)
    y_test = np.asarray(y_test).reshape(-1).astype(int)

    return x_train, x_test, y_train, y_test

In [ ]:
def build_deep_convnet(
    n_classes=4,
    n_channels=64,
    n_samples=512,
    dropout_rate=0.25,
    learning_rate=1e-4,
):
    input_layer = Input(shape=(n_channels, n_samples, 1))

    x = Conv2D(25, (1, 5), kernel_constraint=max_norm(2.0, axis=(0, 1, 2)))(input_layer)
    x = Conv2D(25, (n_channels, 1), kernel_constraint=max_norm(2.0, axis=(0, 1, 2)))(x)
    x = BatchNormalization(epsilon=1e-5, momentum=0.9)(x)
    x = Activation("elu")(x)
    x = MaxPooling2D(pool_size=(1, 2), strides=(1, 2))(x)
    x = Dropout(dropout_rate)(x)

    for filters in [50, 100, 200]:
        x = Conv2D(filters, (1, 5), kernel_constraint=max_norm(2.0, axis=(0, 1, 2)))(x)
        x = BatchNormalization(epsilon=1e-5, momentum=0.9)(x)
        x = Activation("elu")(x)
        x = MaxPooling2D(pool_size=(1, 2), strides=(1, 2))(x)
        x = Dropout(dropout_rate)(x)

    x = Flatten()(x)
    output_layer = Dense(n_classes, kernel_constraint=max_norm(0.5), activation="softmax")(x)

    model = Model(inputs=input_layer, outputs=output_layer)
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [ ]:
# Load data
set_random_seed(SEED)

x_train, x_test, y_train_raw, y_test_raw = load_emg_dataset(
    data_dir=DATA_DIR,
    muscle=MUSCLE,
    n_channels=N_CHANNELS,
    n_samples=N_SAMPLES,
    window_label=WINDOW_LABEL,
)

y_train = to_categorical(y_train_raw - 1, num_classes=N_CLASSES)
y_test = to_categorical(y_test_raw - 1, num_classes=N_CLASSES)

print(f"Muscle: {MUSCLE}")
print(f"Training data shape: {x_train.shape}")
print(f"Testing data shape: {x_test.shape}")
print(f"Training label shape: {y_train.shape}")
print(f"Testing label shape: {y_test.shape}")

In [ ]:
# Train model
model = build_deep_convnet(
    n_classes=N_CLASSES,
    n_channels=N_CHANNELS,
    n_samples=N_SAMPLES,
    dropout_rate=DROPOUT_RATE,
    learning_rate=LEARNING_RATE,
)

model.summary()

history = model.fit(
    x=x_train,
    y=y_train,
    validation_data=(x_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    verbose=1,
)

In [ ]:
# Evaluate the classification performance
probabilities = model.predict(x_test)
y_pred = np.argmax(probabilities, axis=1)
y_true = np.argmax(y_test, axis=1)

overall_accuracy = accuracy_score(y_true, y_pred)
print(f"Overall accuracy for {MUSCLE}: {overall_accuracy:.4f}")

def posture_wise_accuracy(y_true, y_pred):
    rows = []
    for class_idx, posture_name in enumerate(POSTURE_NAMES):
        mask = y_true == class_idx
        acc = accuracy_score(y_true[mask], y_pred[mask]) if np.any(mask) else np.nan
        rows.append({
            "class_index": class_idx,
            "posture": posture_name,
            "n_samples": int(np.sum(mask)),
            "accuracy": acc,
        })
    return pd.DataFrame(rows)

class_accuracy = posture_wise_accuracy(y_true, y_pred)
class_accuracy

In [ ]:
# Save results
prefix = MUSCLE.upper()

summary = pd.DataFrame([{
    "muscle": prefix,
    "overall_accuracy": overall_accuracy,
    "n_train": x_train.shape[0],
    "n_test": x_test.shape[0],
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "dropout_rate": DROPOUT_RATE,
    "seed": SEED,
}])

summary.to_csv(OUTPUT_DIR / f"{prefix}_summary.csv", index=False)
class_accuracy.to_csv(OUTPUT_DIR / f"{prefix}_posture_wise_accuracy.csv", index=False)

print(summary)

## Comparing SOL and FDB

Run this notebook once with `MUSCLE = "SOL"` and once with `MUSCLE = "FDB"`. Then compare the generated files:

- `SOL_summary.csv` vs. `FDB_summary.csv`
- `SOL_posture_wise_accuracy.csv` vs. `FDB_posture_wise_accuracy.csv`